# Phase 5 Frente 3 — LLM caption rewriting

Generates 3 short, grounded captions per training image, using Qwen2.5-7B-Instruct. Total expected: 271 images × 3 captions = 813 rows.

**Runtime:** ~20-30 min on A100. Each LLM call generates 3 captions in one shot.

**Idempotent:** if Colab disconnects mid-way, rerun — already-generated rows are detected and skipped.

## 1. Mount Drive and clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/GabrielmAlves/dgm-2026.1.git"
BRANCH = "feature/generate_datasets"

import os, subprocess
REPO_DIR = "/content/binding-research"
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
%cd $REPO_DIR/projects/challenges-in-attribute-control

## 2. Install deps

In [ ]:
!pip install -q uv
!uv pip install -e . --system
!pip install -q transformers accelerate

## 3. HF login

In [ ]:
from huggingface_hub import login
login()

## 4. Confirm GPU

In [ ]:
import torch
assert torch.cuda.is_available()
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {torch.cuda.get_device_name(0)} ({vram:.1f} GB)')
assert vram >= 16, 'Need A100 or L4 for Qwen2.5-7B'

## 5. Pre-download Qwen2.5-7B-Instruct (text-only, ~14GB)

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download('Qwen/Qwen2.5-7B-Instruct', allow_patterns=['*.json', '*.safetensors', '*.txt', '*.model'])

## 6. Run the recaption pipeline

Outputs `/content/train.csv` with 813 rows (271 images × 3 captions).

In [ ]:
!python experiments/exp5_recaption_llm.py \
    --treated-approved /content/drive/MyDrive/binding-research/finetuning/recolor_treated/approved.csv \
    --treated-root /content/drive/MyDrive/binding-research/finetuning/recolor_treated \
    --control-approved /content/drive/MyDrive/binding-research/finetuning/control_verified/approved.csv \
    --control-root /content/drive/MyDrive/binding-research/finetuning/control_candidates \
    --out-csv /content/train.csv \
    --captions-per-image 3 \
    --max-retries 2

## 7. Quick inspection of the result

In [ ]:
import pandas as pd
df = pd.read_csv('/content/train.csv')
print(f'Total rows: {len(df)}')
print(f'Unique images: {df.image_path.nunique()}')
print(f'Average captions per image: {len(df) / df.image_path.nunique():.2f}')
print()
print('=== By role ===')
print(df.role.value_counts())
print()
print('=== Sample captions (random 10) ===')
print(df.sample(10, random_state=0)[['role','object','color','caption']].to_string())
print()
print('=== Caption length stats (words) ===')
df['n_words'] = df.caption.str.split().str.len()
print(df.n_words.describe())
print()
print('=== Images with fewer than 3 captions ===')
incomplete = df.groupby('image_path').size()
incomplete = incomplete[incomplete < 3]
print(f'{len(incomplete)} images have fewer than 3 captions')

## 8. Save train.csv to Drive

In [ ]:
import shutil
DEST = '/content/drive/MyDrive/binding-research/finetuning/train.csv'
shutil.copy('/content/train.csv', DEST)
print(f'Saved to {DEST}')